# 2048 Deep Q-Network
Training and evaluation notebook for the DQN agent.  
Works on **Google Colab** and **Kaggle** out of the box — no extra installs needed.

## 1 · Setup
Clones the repo on first run; pulls latest changes on every subsequent run.  
Also clears the Python module cache so updated code is always picked up without restarting the kernel.

In [ ]:
import os, sys

if os.path.isdir('RL2048'):
    !git -C RL2048 pull
else:
    !git clone https://github.com/<your-username>/RL2048.git

if 'RL2048' not in sys.path:
    sys.path.insert(0, 'RL2048')

# Evict any previously imported local modules from the cache.
# This forces Python to re-read the files from disk after a git pull,
# so you never get stale imports without restarting the kernel.
for _mod in ['game', 'dqn', 'agent', 'environment', 'episode']:
    sys.modules.pop(_mod, None)

In [ ]:
import numpy as np
from game import TwntyFrtyEight
from dqn import (
    DQNAgent,
    plot_training_curve,
    plot_tile_distribution,
    compare_agents,
    _run_one,
)

## 2 · Train

**Architecture & algorithmic improvements (from Li & Peng 2021):**
- **CNN encoder** — two Conv2d layers process the `(16, 4, 4)` one-hot board spatially before the MLP head. This lets the network learn tile-adjacency patterns (corner stacking, monotone rows, merge opportunities) that a flat MLP cannot represent.
- **Double DQN** — the online net selects the best next action; the target net evaluates it. Decoupling selection from evaluation reduces Q-value overestimation, which was causing the agent to plateau at 256/512.
- **Hard target update every 1000 steps** — instead of a soft τ update every step, the target network is copied wholesale every 1000 gradient steps. This gives the network a stable regression target during early training.
- **Reward = raw merge score only** — removing the max-tile bonus gives a clean, stationary signal aligned with the actual game objective.
- **Gradient clipping (max norm 10)** — prevents exploding updates from large merge-score rewards.

**GPU settings (recommended for Kaggle T4 x2 / Colab GPU):**
- `n_envs=16` — 16 parallel game environments; all boards are batched into one GPU forward pass per step.
- `hidden_size=256` — wider MLP head after the conv layers.
- `buffer_capacity=100_000` — larger buffer for better sample diversity.
- `batch_size=512` — larger gradient batches keep the GPU busier.
- `n_games=15000`, `epsilon_decay=0.9998` — slow decay over many games; epsilon hits its floor around episode 8000.

**CPU / quick-test settings:** `n_envs=1, hidden_size=256, n_games=3000, batch_size=128`.

In [ ]:
agent = DQNAgent(
    lr=1e-4,
    hidden_size=256,
    buffer_capacity=100_000,
    target_update_freq=1000,   # hard copy every 1000 optimisation steps
)

train_stats = agent.train(
    n_games=15_000,
    epsilon_start=0.3,
    epsilon_end=0.05,
    epsilon_decay=0.9998,      # hits floor ~ep 8000 with n_games=15000
    batch_size=512,
    print_every=100,
    n_envs=16,                 # parallel envs → batched GPU forward passes
)

In [ ]:
# Save weights so you can reload without retraining
agent.save('dqn_weights.pth')

## 3 · Training curve
Three panels: moves per game, total reward per game, and max tile per game.  
An upward trend in moves = the agent is learning to stay alive longer.

In [ ]:
plot_training_curve(train_stats, window=200)

## 4 · Evaluate
Play greedily (ε = 0) for `n_games` games and collect tile/step stats.

In [ ]:
eval_stats = agent.evaluate(n_games=200)
plot_tile_distribution(eval_stats, label='DQN')

## 5 · Compare agents
Run the SARSA and beam search policies for comparison.  

> **Beam search is slow** — each game at `depth=20, k=10` takes a few seconds.  
> Use `depth=10, k=5` for a quicker run, or reduce `n_games`.

In [ ]:
from agent import Agent

game  = TwntyFrtyEight()
sarsa = Agent(game)

# Load trained SARSA weights if available; otherwise uses zero weights (random-ish)
try:
    loaded = np.load('RL2048/w_star.npy')
    if loaded.shape == sarsa.w.shape:
        sarsa.w = loaded
        print(f'Loaded SARSA weights  shape={loaded.shape}')
    else:
        print('Shape mismatch — using zero weights')
except FileNotFoundError:
    print('w_star.npy not found — using zero weights')

In [ ]:
N_SARSA    = 100   # greedy SARSA games
N_BEAM     = 30    # beam search games (slow — keep small)
BEAM_DEPTH = 10
BEAM_K     = 5

print(f'Running {N_SARSA} SARSA games…')
sarsa_stats = [_run_one(game, sarsa.greedy_policy) for _ in range(N_SARSA)]

print(f'Running {N_BEAM} beam search games (depth={BEAM_DEPTH}, k={BEAM_K})…')
beam_stats = [
    _run_one(game, lambda s: sarsa.beam_search_policy(s, depth=BEAM_DEPTH, k=BEAM_K))
    for _ in range(N_BEAM)
]

print('Done.')

In [ ]:
compare_agents({
    'DQN':                               eval_stats,
    'SARSA':                             sarsa_stats,
    f'Beam (d={BEAM_DEPTH},k={BEAM_K})': beam_stats,
})

## 6 · Resume training (optional)
Load saved weights and continue training from where you left off.

In [ ]:
agent2 = DQNAgent(hidden_size=256, buffer_capacity=100_000)
agent2.load('dqn_weights.pth')

more_stats = agent2.train(
    n_games=5000, epsilon_start=0.1, batch_size=512, n_envs=16, print_every=100
)
agent2.save('dqn_weights.pth')

# Plot combined training history
all_stats = train_stats + [
    {**s, 'game': s['game'] + len(train_stats)} for s in more_stats
]
plot_training_curve(all_stats, window=200)